In [1]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '_detected_manual' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [2]:
# Parameters
query_ratio = 0.2
seed = 2


In [3]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict
import sys
sys.path.append(r'C:\BP\pythonProject1')
from misclassification_utils import show_misclassified

In [4]:
np.random.seed(seed)

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (315, 768)


In [5]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# optional: keep full dataset tensors for evaluation later
X = torch.tensor(embeddings, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# build training dataset and loader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [7]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
# determine number of classes from training labels
num_classes = len(torch.unique(torch.tensor(y_train)))

model = Classifier(input_dim=768, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=37.745, acc=0.250
Epoch 1: loss=28.721, acc=0.444
Epoch 2: loss=23.113, acc=0.544
Epoch 3: loss=18.581, acc=0.643
Epoch 4: loss=13.741, acc=0.774
Epoch 5: loss=12.254, acc=0.762
Epoch 6: loss=7.868, acc=0.853
Epoch 7: loss=6.824, acc=0.885


Epoch 8: loss=5.511, acc=0.893
Epoch 9: loss=4.572, acc=0.905
Epoch 10: loss=4.319, acc=0.921
Epoch 11: loss=3.526, acc=0.948
Epoch 12: loss=2.311, acc=0.960
Epoch 13: loss=2.332, acc=0.972
Epoch 14: loss=1.660, acc=0.964
Epoch 15: loss=1.078, acc=0.988


Epoch 16: loss=1.197, acc=0.988
Epoch 17: loss=1.129, acc=0.980
Epoch 18: loss=1.127, acc=0.976
Epoch 19: loss=0.818, acc=0.984
Epoch 20: loss=0.648, acc=0.996
Epoch 21: loss=0.574, acc=0.996
Epoch 22: loss=0.581, acc=0.992
Epoch 23: loss=0.695, acc=0.988


Epoch 24: loss=0.738, acc=0.992
Epoch 25: loss=0.526, acc=0.996
Epoch 26: loss=0.638, acc=0.992
Epoch 27: loss=0.977, acc=0.976
Epoch 28: loss=0.358, acc=0.992
Epoch 29: loss=0.617, acc=0.996
Epoch 30: loss=0.536, acc=0.996
Epoch 31: loss=0.442, acc=0.992
Epoch 32: loss=0.588, acc=0.980


Epoch 33: loss=0.457, acc=0.988
Epoch 34: loss=0.395, acc=0.996
Epoch 35: loss=0.233, acc=1.000
Epoch 36: loss=0.267, acc=0.996
Epoch 37: loss=0.283, acc=0.996
Epoch 38: loss=0.430, acc=0.992
Epoch 39: loss=0.298, acc=0.992
Epoch 40: loss=0.352, acc=0.992
Epoch 41: loss=0.469, acc=0.988


...

Epoch 60: loss=0.231, acc=0.996
Epoch 61: loss=0.261, acc=0.992
Epoch 62: loss=0.242, acc=0.996
Epoch 63: loss=0.364, acc=0.988
Epoch 64: loss=0.423, acc=0.988
Epoch 65: loss=0.307, acc=0.996
Epoch 66: loss=0.182, acc=0.996
Epoch 67: loss=0.130, acc=0.996
Epoch 68: loss=0.163, acc=0.996


Epoch 69: loss=0.187, acc=0.996
Epoch 70: loss=0.133, acc=0.996
Epoch 71: loss=0.119, acc=0.996
Epoch 72: loss=0.128, acc=0.996
Epoch 73: loss=0.324, acc=0.988
Epoch 74: loss=0.477, acc=0.992
Epoch 75: loss=0.471, acc=0.988
Epoch 76: loss=0.561, acc=0.988
Epoch 77: loss=0.311, acc=0.988


Epoch 78: loss=0.309, acc=0.996
Epoch 79: loss=0.175, acc=0.996
Epoch 80: loss=0.125, acc=1.000
Epoch 81: loss=0.313, acc=0.992
Epoch 82: loss=0.165, acc=0.996
Epoch 83: loss=0.335, acc=0.988
Epoch 84: loss=0.215, acc=0.992
Epoch 85: loss=0.165, acc=0.992
Epoch 86: loss=0.225, acc=0.996


Epoch 87: loss=0.170, acc=0.996
Epoch 88: loss=0.110, acc=1.000
Epoch 89: loss=0.092, acc=0.996
Epoch 90: loss=0.412, acc=0.992
Epoch 91: loss=0.541, acc=0.988
Epoch 92: loss=0.238, acc=0.996
Epoch 93: loss=0.144, acc=1.000
Epoch 94: loss=0.177, acc=0.996
Epoch 95: loss=0.051, acc=1.000


Epoch 96: loss=0.188, acc=0.996
Epoch 97: loss=0.150, acc=0.996
Epoch 98: loss=0.091, acc=1.000
Epoch 99: loss=0.151, acc=0.996


In [9]:
# Evaluate on training set
model.eval()
with torch.no_grad():
    preds = model(X_train_tensor).argmax(1)
    accuracy = (preds == y_train_tensor).float().mean()
    print("Final train accuracy:", accuracy.item())

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    print("Final train loss:", loss.item())

Final train accuracy: 0.9960317611694336
Final train loss: 0.005692599341273308


In [10]:
# Check misclassified samples on training set
show_misclassified(y_train_tensor, preds, idx_train, detection, encoder)

Number wrong: 1
Index 134 (Orig 110): Brano\Dio_4.JPG
  True: Brano, Predicted: Dio



## CrossEntropy Loss


In [11]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    preds_test = model(X_test_tensor).argmax(1)
    accuracy_test = (preds_test == y_test_tensor).float().mean()
    print("Final validation accuracy:", accuracy_test.item())

    outputs_test = model(X_test_tensor)
    loss_test = criterion(outputs_test, y_test_tensor)
    print("Final validation loss:", loss_test.item())

Final validation accuracy: 0.6349206566810608
Final validation loss: 2.4027366638183594


In [12]:
# reuse helper function defined earlier to list misclassified samples on validation set
show_misclassified(y_test_tensor, preds_test, idx_test, detection, encoder)

Number wrong: 23
Index 0 (Orig 249): Milos\Milos_34.JPG
  True: Milos, Predicted: Lubos

Index 7 (Orig 285): Roman\Roman_23.JPG
  True: Roman, Predicted: Edo

Index 10 (Orig 302): Silvester\Silvester_4.jpg
  True: Silvester, Predicted: Edo

Index 14 (Orig 104): Brano\Brano_3.JPG
  True: Brano, Predicted: Kiara

Index 15 (Orig 35): Albin\Albin_4.JPG
  True: Albin, Predicted: Zora

Index 18 (Orig 29): Albin\Albin_33.JPG
  True: Albin, Predicted: Zora

Index 23 (Orig 159): Eliska\Eliska_6.JPG
  True: Eliska, Predicted: Milos

Index 24 (Orig 114): Dio\Dio_13.jpg
  True: Dio, Predicted: Silvester

Index 25 (Orig 30): Albin\Albin_35.JPG
  True: Albin, Predicted: Benadik

Index 29 (Orig 154): Eliska\Eliska_11.JPG
  True: Eliska, Predicted: Brano

Index 30 (Orig 141): Edo\Edo_26.JPG
  True: Edo, Predicted: Eliska

Index 31 (Orig 178): Izidor\Izidor_23.JPG
  True: Izidor, Predicted: Milos

Index 32 (Orig 184): Izidor\Izidor_29.JPG
  True: Izidor, Predicted: Roman

Index 36 (Orig 289): Roman\Rom

## Poznamenanie k výsledkom tréningu

- **Izidor_27** (nočná fotka zozadu) bol nesprávne klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov zozadu, ale aj veľa nočných.
- **Eliška_7** sa pravdepodobne podobá na **Braňa**.
- **Kiara_17** je nočný dobre osvetlený záber zboku s kontrastným zatmeným pozadím, veľmi podobný mnohým zaberom **Romana** s týmito charakteristikami.
- **Izidor_26** je záber zboku s výnimočne zeleným pozadím, nesprávne klasifikovaný ako **Roman**, ktorý má v datasete (v porovnaní s ostatnými) výrazne veľa snímok zboku.
- **Zora_5** bola pre kombináciu sneh + ihličnany klasifikovaná ako **Izidor**, ktorý má v tréningovom sete veľa obrázkov tohto typu.
- **Izidor_37** (nočná fotka + svietiace oči) bol klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov s touto kombináciou.
- **Brano_2** (jesenná fotka) bol nesprávne klasifikovaný ako **Eliška**, u ktorej sú niektoré jesenné obrázky.

In [13]:
result = {
    "megadescriptor_version": megadescriptor_version,
    "dataset_version": detection,
    "seed": seed,
    "query_ratio": query_ratio,
    "accuracy": accuracy_test.item(),
    "loss_function": "CrossEntropyLoss"
}

result

{'megadescriptor_version': 'T-224',
 'dataset_version': '_detected_manual',
 'seed': 2,
 'query_ratio': 0.2,
 'accuracy': 0.6349206566810608,
 'loss_function': 'CrossEntropyLoss'}